# 01 Bronze: file manifest

| | |
|---|---|
| **Layer** | Bronze |
| **Reads** | PDF files under `Files/bronze/unstructureddata` |
| **Writes** | Delta table `bronze_doc_manifest` |
| **Re-runnable (idempotent)** | Yes. Only files with a new SHA-256 hash are inserted. |

Bronze keeps the raw files exactly as received and never edits them. This notebook adds a control table (the manifest) that records one row per file: its content hash, name, path, size, and when it was first loaded. The hash is the document identity used for lineage in every later layer. An edited file gets a new hash, so it is treated as a new document.

**Standards followed:** parameters at the top, no secrets in code, small documented functions, explicit schemas, a validation cell at the end, and a printed run summary.

## Parameters

In [ ]:
SOURCE_FOLDER   = "/lakehouse/default/Files/bronze/unstructureddata"   # raw, immutable input files
LAKEHOUSE_ROOT  = "/lakehouse/default"                                 # used to build Spark-relative paths
MANIFEST_TABLE  = "bronze_doc_manifest"
FILE_SUFFIX     = ".pdf"

## Imports

In [ ]:
import os, hashlib, uuid
from datetime import datetime
from pyspark.sql import functions as F

## Helpers

In [ ]:
MANIFEST_SCHEMA = ("sha256 string, file_name string, file_path string, "
                   "size_bytes long, load_ts timestamp, batch_id string")


def sha256_of(path: str, block_size: int = 1 << 20) -> str:
    """Return the SHA-256 hex digest of a file, read in blocks so large files do not fill memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def scan_folder(root: str, suffix: str, batch_id: str, load_ts: datetime) -> list:
    """Walk the folder and return one tuple per file matching the suffix."""
    rows = []
    for dirpath, _, names in os.walk(root):
        for name in sorted(names):
            if not name.lower().endswith(suffix):
                continue
            full = os.path.join(dirpath, name)
            rows.append((
                sha256_of(full),
                name,
                os.path.relpath(full, LAKEHOUSE_ROOT),      # e.g. Files/bronze/unstructureddata/x.pdf
                os.path.getsize(full),
                load_ts,
                batch_id,
            ))
    return rows

## Load: insert only files not seen before
A left anti join on the hash makes reruns safe. Without it, every rerun would append duplicate manifest rows.

In [ ]:
batch_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
found = spark.createDataFrame(scan_folder(SOURCE_FOLDER, FILE_SUFFIX, batch_id, datetime.utcnow()),
                              MANIFEST_SCHEMA)

if spark.catalog.tableExists(MANIFEST_TABLE):
    known = spark.table(MANIFEST_TABLE).select("sha256").distinct()
    new_files = found.join(known, "sha256", "left_anti")        # keep only unseen hashes
else:
    new_files = found

new_count = new_files.count()
if new_count:
    new_files.write.format("delta").mode("append").saveAsTable(MANIFEST_TABLE)

print(f"batch={batch_id} | files found={found.count()} | new files loaded={new_count}")

## Validation
Expected: no hash appears twice in the manifest. A file copied under two names shares one hash and is reported as a duplicate in notebook 02.

In [ ]:
manifest = spark.table(MANIFEST_TABLE)
dupes = manifest.groupBy("sha256").count().filter("count > 1")
print("rows:", manifest.count(), "| distinct documents:", manifest.select("sha256").distinct().count(),
      "| hashes appearing more than once:", dupes.count())
display(manifest.orderBy("file_name").limit(20))